# 02 — What is happening during the run, in triplicate
Journals (and good practice) want **independent repeats**. We run three trajectories from the same minimized structure with different initial velocities (seeds 2024/2025/2026), look at the first in detail, then overlay all three to see the spread you should expect. The *average* behaviour is consistent across runs, even though individual events (a hydrogen bond breaking, a dihedral wandering) happen at different times in each. Across independent repeats, MD reproducibility is statistical: the averages repeat, the frame-by-frame history does not.

**Two ways to run this notebook.** *Live mode* (the default): run short simulations from the system `01_build_system` prepared. If that saved system is absent, or you are on Colab, the notebook prepares its own and says so. *Reference mode* (`LOAD_REFERENCE = True`): skip the simulation and analyze the longer precomputed reference trajectories shipped with the tutorial. No GPU needed. Details are in the README.

**Reproducibility, in one line.** Bit-for-bit repeats are possible but need care (a fixed prepared system, a fresh Context per run, CUDA with `DeterministicForces`, and the same GPU model). Across machines you rely on **statistical** reproducibility, which is the seed-to-seed spread the three repeats show. `determinism.ipynb` takes this apart properly.

> **Colab users:** set **Runtime > Change runtime type > T4 GPU** *before* running anything, and set it again for **every** notebook you open. Each notebook runs in its own VM, so the setting does not carry over. Cell 0 prints a warning if it finds no GPU.

> **Terms used in this notebook** (full glossary in [`00_intro`](https://github.com/todd471/MD_tutorial/blob/main/00_intro.md#glossary); terms from `01_build_system` are assumed)
>
> - **Trajectory, frame** · The saved sequence of coordinate snapshots from a run. One snapshot is a frame (every 1 ps here).
> - **Integrator, timestep** · The algorithm that advances every atom by one small time step (2 fs here) using the current forces.
> - **Seed** · The number that fixes the random draws (initial velocities, thermostat noise). Same seed on the same hardware repeats a run; a different seed gives an independent repeat.
> - **NVE, NVT, NPT** · Ensemble labels. N = number of atoms, V = volume, E = total energy, T = temperature, P = pressure; the letters name what is held fixed. NVE: energy and volume constant (plain Newton). NVT: temperature and volume constant (what these notebooks run). NPT: temperature and pressure constant, so the box resizes.
> - **Thermostat, barostat** · The algorithm that holds the *average* temperature (thermostat) or pressure, by resizing the box (barostat). Langevin is the thermostat used here.
> - **Observable** · Any number computed from a frame: a distance, an angle, an energy, a surface area. Observables are the measurements of a simulation.
> - **Collective variable (CV)** · An observable chosen to summarize one motion of interest in a single number, for example the distance between two groups of atoms. Used to track a motion here, and to push on it in `03_enhanced_sampling`.
> - **RMSD, Cα RMSD** · Root-mean-square deviation: after overlaying a frame on a reference structure, the average distance between matching atoms. Using only the backbone Cα atoms gives a fold-level measure. Small means still close to the reference fold.
> - **Radius of gyration (Rg)** · The root-mean-square distance of the atoms from the molecule's center. A compact fold gives a small Rg, an unfolded chain a large one.
> - **Dihedral, rotamer, well** · A dihedral (torsion) is the rotation angle about a bond: χ1 for a side chain, φ and ψ for the backbone. Side chains prefer a few torsion ranges called rotamers. A *well* is a region of low energy the system settles into; a rotamer is one such well, and the *folded well* is the set of near-native conformations.
> - **Helix fraction** · The fraction of residues assigned as α-helix in a frame (by the DSSP algorithm).
> - **Salt bridge** · A close contact between an acidic and a basic side chain, here ASP9 and ARG16.
> - **Integrated autocorrelation time (τ)** · How long an observable takes to forget its earlier value. Frames closer together than about 2τ are not independent samples, so a run of length L holds only about L / 2τ independent measurements of that observable. It is the number that decides whether a run is long enough.
> - **Block averaging** · Splitting a run into blocks and comparing the block means. The spread between blocks stops growing with block size only once blocks are longer than τ, which is how τ is estimated.
> - **Scalogram** · A map of how strongly an observable fluctuates at each timescale, across the run (§2.6b).

In [ ]:
#@title Environment on-ramp (imports + modules)
# --- environment on-ramp: make sure the MD stack + the module are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
if "google.colab" in sys.modules:                                       # Colab: is this runtime actually a GPU one?
    import openmm as _mm, shutil as _sh                                 # (the pip wheel may expose the T4 as OpenCL, not CUDA)
    _gpu = {"CUDA", "OpenCL"} & {_mm.Platform.getPlatform(i).getName() for i in range(_mm.Platform.getNumPlatforms())}
    if not _gpu and _sh.which("nvidia-smi"):
        print("!" * 78 + "\n!!  A GPU is attached, but OpenMM loaded no CUDA/OpenCL platform (plugin load failures below).\n!!  "
              + str(_mm.Platform.getPluginLoadFailures())[:300] + "\n" + "!" * 78)
    elif not _gpu:
        print("!" * 78 + "\n!!  This Colab runtime has NO GPU. Runtime > Change runtime type > T4 GPU, then\n"
              "!!  Runtime > Run all. (CPU works, but explicit-solvent MD is painfully slow.)\n" + "!" * 78)
    else:
        print(f"Colab GPU runtime OK: OpenMM will use {sorted(_gpu)[0]}.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/todd471/MD_tutorial/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "md_scalogram.py"):          # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import mdtutorial as mdt, mdtviz
PYMOL = mdtviz.setup_pymol()                                            # find/provision headless PyMOL (panels skip if none)

In [ ]:
# --- configuration ---
import numpy as np, mdtraj as md
import matplotlib.pyplot as plt
OUT  = "trpcage_out"   # scratch root: prep, figures, trajectories. Local -- on Colab it's the VM's ephemeral disk.
PREP = OUT             # where 01 SAVES its prepared system and 02/03 LOAD it. LOCAL/standalone: one shared
                       # filesystem, so 02/03 load 01's EXACT prepared system for free. COLAB: each notebook runs
                       # in its own VM with no shared disk, so when 01's prep isn't here 02/03 loudly re-prepare an
                       # INDEPENDENT system (see the load cell) -- fine for dynamics, but a different solvation than
                       # 01's, so NOT bit-identical to it. To share 01's exact prep on Colab, point PREP at a Google
                       # Drive path instead (opt-in; see the README's Colab section).
N_PROD_PS = int(os.environ.get("N_PROD_PS", "200"))   # production length per repeat (ps); override for a quick test
SEEDS     = [2024, 2025, 2026] # independent repeats -- EDIT this list to change which/how many (the run count
                               #   is just len(SEEDS)). For the 6-seed canonical reference use the six-seed list:
                               #   [2021, 2022, 2023, 2024, 2025, 2026].
RUN_MODE  = "interactive"      # "interactive" (default) = trajectory + state log only. "canonical" ALSO writes the archival slate (checkpoints/system.xml/integrator/
                               #   final_state/run_meta) to OUT -- set it with the six-seed SEEDS above +
                               #   N_PROD_PS=10000 to GENERATE the reference. Writes to OUT only; never touches REF_ROOT.
REF_ROOT  = "reference_10ns"   # SHIPPED read-only reference (6 seeds x 10 ns); §2.6/timescales load it for a meaningful
                               #   autocorrelation time. Separate from OUT, so a canonical rerun never overwrites it.
LOAD_REFERENCE = False         # False = live tier (load 01's prep + simulate). True = analyze the REF_ROOT
                               #   trajectories instead (no GPU, no 01): the same figures on multi-ns runs.
CANON_SEED = 2025              # example seed for the §2.6 timescale + §2.6b scalogram (2025: ILE4 χ1 hops out to t
                               #   and back, several salt-bridge breaks, and a clean fast->slow χ1 strip).
PLOT_SEEDS = [2022, 2025, 2026]  # which seeds to DRAW in the §2.5 overlay -- ALL loaded seeds are still analyzed;
                               #   this only declutters the plot. Edit to show any subset (the other seeds are there).

### 2.0  The force field: the potential behind every motion
Everything you are about to watch happens on a single potential energy function **U(x)**, the *force field*. The forces pull each atom downhill on that surface, but the atoms are moving (they carry kinetic energy at 300 K), so they coast back uphill and over barriers as well; a thermostat, when one is used, only keeps that kinetic energy at the target temperature. The peptide explores the surface rather than sliding to its bottom. U(x) is a sum of simple terms: springs for **bonds** and **angles** (a stretched bond costs energy like a stretched spring), a periodic function for **dihedral torsions** (rotation about a bond has preferred angles), and pairwise **Lennard-Jones + Coulomb** for everything **non-bonded** (atoms repel when they overlap, attract weakly at short range, and charges interact). Several of the observables below follow one of these terms directly: the **dihedral** panel watches an ILE4 χ1 torsion, and the **H-bond / salt-bridge** distances track non-bonded contacts. Bonds and angles are so stiff they blur out at this timescale, which is why the coordinates worth watching are the softer torsional and non-bonded ones.

In [ ]:
#@title §2.0 — force-field energy-terms figure (code)
# U(x) = sum of these terms; the CVs below each report on one of them. Colors match the paper's
# force-field figure (turbo hue of the residue where each term is illustrated).
_turbo = plt.get_cmap("turbo")
def _tc(resi, nres=20):                                     # turbo hue for a residue, darkened enough to read as text
    r, g, b, _ = _turbo((resi - 1) / (nres - 1))
    lum = 0.299 * r + 0.587 * g + 0.114 * b; f = 0.55 / lum if lum > 0.55 else 1.0
    return (r * f, g * f, b * f)
_TERMS = [   # (title, TeX, residue-for-color, what it governs in THIS notebook)
    ("Bond stretch",     r"$\sum_{\mathrm{bonds}} k_b\,(b-b_0)^2$",                  19, "stiff & fast — blurs out at this timescale"),
    ("Angle bend",       r"$\sum_{\mathrm{angles}} k_\theta\,(\theta-\theta_0)^2$",  16, "stiff & fast — blurs out at this timescale"),
    ("Non-bonded  (LJ + Coulomb)", r"$\sum_{i<j}\varepsilon\left[(r_m/r)^{12}-2(r_m/r)^{6}\right]+\dfrac{q_iq_j}{\varepsilon_r r}$", 9, "→ GLN5/ASP9 H-bonds & ASP9–ARG16 salt bridge (§2.5)"),
    ("Dihedral torsion", r"$\sum_{\mathrm{dih.}} k_\phi\left[1+\cos(n\phi-\delta)\right]$",  4, "→ ILE4 χ1 rotamer hop (§2.5)"),
]
_ff, _axs = plt.subplots(2, 2, figsize=(11, 3.7))
_ff.suptitle(r"The force field:  $U(\mathbf{x})=\sum_{\mathrm{bonded}}+\sum_{\mathrm{non\text{-}bonded}}$  —  the potential the dynamics roll on",
             fontsize=11, y=1.02)
for _ax, (_ttl, _eq, _resi, _gov) in zip(_axs.ravel(), _TERMS):
    _c = _tc(_resi); _ax.axis("off")
    _ax.text(0.02, 0.82, _ttl, fontsize=12, fontweight="bold", color=_c, va="center")
    _ax.text(0.05, 0.46, _eq, fontsize=15, color=_c, va="center")
    _ax.text(0.05, 0.10, _gov, fontsize=9, style="italic", color="0.35", va="center")
_ff.tight_layout(); plt.show()

### Get the trajectories — live tier (simulate) or reference tier (load)
Default (*live tier*): load `01_build_system`'s prepared system and simulate. `01_build_system` owns preparation; this notebook **loads** what it saved if present, and otherwise **prepares its own on the spot** (`load_or_prepare` prints which path it took, so it is never silent). That fallback prep is an *independent* solvation (this notebook's default seed, not `01_build_system`'s), fine for dynamics but not for reproducing `01`'s exact run bit-for-bit. With **`LOAD_REFERENCE = True`** (*reference tier*): load the shipped multi-ns trajectories from `REF_ROOT` instead: no prep, no GPU. Either way, everything below just reads `cvs`.

In [ ]:
if LOAD_REFERENCE:
    mdt.ensure_reference(REF_ROOT, _BASE)                   # Colab: pull the bundle into this VM (no-op if already local)
    cvs_all, RUN_SEEDS, DT = mdt.load_reference(REF_ROOT)   # analyze the shipped long run: no GPU, no 01
    prep = None
    print(f"reference tier: {len(cvs_all)} seeds x {DT * cvs_all[0]['t'].n_frames / 1000:g} ns from {REF_ROOT}/")
else:
    prep = mdt.load_or_prepare(PREP, OUT)                  # reuse 01's prep if present, else prep a fresh one here (self-sufficient)
    cvs_all, RUN_SEEDS, DT = None, SEEDS, 1.0

### 2.1  The first trajectory
*Live tier:* a fresh Context (so its random stream is reproducible) writes a trajectory + a scalar state log, and we compute the collective variables from it. *Reference tier:* we take the first shipped trajectory. Either way we look at this one in detail before overlaying the rest.

*Code: `run_repeat` (in `mdtutorial.py`) runs one trajectory in a **fresh** OpenMM Context with a `LangevinMiddleIntegrator` (the LFMiddle scheme; Zhang *et al.* 2019); a fresh Context per run is what makes the seed reproducible; `compute_cvs` derives the observables with **MDTraj**.*

In [ ]:
if LOAD_REFERENCE:
    cvs = [cvs_all[0]]
else:
    cvs = [mdt.compute_cvs(mdt.run_repeat(prep, RUN_SEEDS[0], n_prod_ps=N_PROD_PS, run_mode=RUN_MODE, out_root=OUT),
                           top=mdt.outp("stage4_minimized.pdb", PREP))]
print(f"seed {RUN_SEEDS[0]}: RMSD end {cvs[0]['rmsd'][-1]:.1f} A, helix {cvs[0]['helix'].mean():.2f}")

### The words behind the run: *ensemble*, *thermostat*, *barostat*
The vital-signs panels below are labelled **NVT**, **thermostat**, **canonical**. Thirty seconds of vocabulary first, because these terms are everywhere in MD and rarely defined.

In statistical mechanics, an **ensemble** names which bulk properties are held constant while the atoms move:

- N = number of particles
- V = volume
- T = temperature
- P = pressure
- E = total energy

So **NVE** holds particle number, volume, and energy constant; **NVT** holds particle number, volume, and temperature constant (the *canonical* ensemble); **NPT** holds particle number, pressure, and temperature constant, so the box resizes to keep the pressure. Plain Newtonian dynamics is NVE: it conserves energy. A protein in a test tube is closer to NPT: it sits in a bath at roughly constant temperature and pressure, trading energy with its surroundings. You pick the ensemble to match the conditions you want to **model** (a solvated protein near physiological temperature and about 1 atm), not the conditions the structure was solved in (a crystal or a vitrified cryo-EM grid is essentially a solid). In practice the choice is also **staged**: NPT first, so the freshly built box can relax to the right density, then production under the ensemble that represents the target state, with NVE useful as an energy-conservation check. This pipeline does the same in miniature: `01_build_system` runs 20 ps NVT and then 100 ps NPT, and production is NVT at the relaxed box. The sandbox lets you flip NPT on and watch the density settle for yourself.

Newton's equations cannot hold temperature fixed on their own, because they conserve energy, not T. A **thermostat** supplies the missing heat bath. It adds or removes small amounts of kinetic energy to keep the *average* temperature on target while the energy fluctuates as it physically should. A **barostat** does the same for pressure by rescaling the box. That is why, below, the **total energy is _not_ flat under NVT**: it is being exchanged with the thermostat, while the temperature holds steady on average and the box volume stays fixed (NVT, no barostat). *Which* thermostat is itself a real choice (you flip it in the sandbox). Methods differ in how faithfully they reproduce the fluctuations, and some (Berendsen) are fine for settling a system but quietly wrong for production.

### 2.1c  What a run writes to disk
A run leaves **files** on disk, not just a plot (whether you simulated it here or loaded the shipped reference). MD writes **two kinds of data**, and every figure later in this notebook pulls from one of them:

| output | what it is | you read it for |
|---|---|---|
| **trajectory** (`traj_<seed>.dcd`) | atomic **coordinates** (every ps in a live run) | structure and motion: RMSD, Rg, H-bonds, the animation (all of §2.2 onward) |
| **state log** (`state_<seed>.csv`) | scalar **thermodynamics** every ps: potential/kinetic/total energy, temperature, box volume, density | the run's *vital signs*: did it equilibrate, is the thermostat holding, is the box sane |
| *(`RUN_MODE = "canonical"` only, the full archival output; a software setting, not the ensemble)* `checkpoint_*.chk`, `system.xml`, `integrator_<seed>.xml`, `final_state_*.xml` | the exact **inputs + restart state** | extending or reproducing the run bit-for-bit (see `determinism.ipynb`) |

The trajectory gets all the attention, but the **state log is your sanity check**. After a run (or live, by tailing the file on a long cluster job) it tells you whether the coordinates you are about to analyze came from a physically sane, equilibrated system rather than one slowly cooking or exploding. Our runner starts from the equilibrated system `01_build_system` saved and records production only *after* a further 20 ps NVT settle from freshly drawn velocities. A healthy log is **not** a flat line. It is **noisy but stationary**: the *mean* holds steady while the instantaneous values scatter around it. That scatter is real thermal fluctuation, and it is large here because the system is small (a few thousand atoms). The temperature swings about ±4 K around 300 K, which matches the canonical fluctuation σ_T = T·√(2⁄N_dof) for a system this size (Lebowitz *et al.* 1967; Frenkel & Smit), and the potential energy ±0.4%. Stability is the flat *mean* (the run below drifts < 0.05% end-to-end), not a flat trace.

In [ ]:
#@title §2.1c — what a run writes to disk (code)
_md_dir = os.path.join(REF_ROOT if LOAD_REFERENCE else OUT, "md_output")   # live run, or the reference bundle
print("files in", _md_dir + "/ :")
for _f in sorted(os.listdir(_md_dir)):                     # what MD actually produced (trajectory + state log)
    print(f"    {_f:26s} {os.path.getsize(os.path.join(_md_dir, _f)) / 1e6:8.2f} MB")
D = np.genfromtxt(os.path.join(_md_dir, f"state_{RUN_SEEDS[0]}.csv"), delimiter=",", skip_header=1)   # step,time,PE,KE,E,T,vol,rho,speed
t, pe, te, temp, vol = D[:, 1] / 1000.0, D[:, 2], D[:, 4], D[:, 5], D[:, 6]
_drift = np.polyfit(t, pe, 1)[0] * (t[-1] - t[0]) / abs(pe.mean()) * 100   # % change of the PE mean over the whole run
fig, ax = plt.subplots(2, 2, figsize=(10, 6.2), sharex=True, constrained_layout=True)
panels = [(ax[0, 0], pe,   "C0", "kJ/mol",  f"potential energy — mean {pe.mean():.0f} (±{pe.std()/abs(pe.mean())*100:.1f}% thermal)"),
          (ax[0, 1], temp, "C3", "K",       f"temperature — {temp.mean():.1f} ± {temp.std():.1f} K"),
          (ax[1, 0], te,   "C2", "kJ/mol",  "total energy — NVT (exchanged w/ thermostat, not conserved)"),
          (ax[1, 1], vol,  "C4", "nm$^3$",  "box volume — flat (NVT, no barostat)")]
for a, y, col, yl, ttl in panels:
    a.plot(t, y, lw=0.5, color=col)
    a.axhline(y.mean(), color="k", ls="--", lw=1.2)        # the MEAN: flat mean = stable; the scatter around it is thermal
    a.set_title(ttl, fontsize=9.5); a.set_ylabel(yl)
ax[0, 1].set_ylim(temp.mean() - 20, temp.mean() + 20)      # ±20 K context so the ±4 K thermal swing reads as small
ax[1, 1].set_ylim(vol.mean() - 0.06, vol.mean() + 0.06)    # tight window: a constant NVT volume plots as the flat line it is
ax[1, 0].set_xlabel("time (ns)"); ax[1, 1].set_xlabel("time (ns)")
fig.suptitle("Run vital signs — the MEAN holds; the scatter is expected thermal noise (small system)", fontsize=12)
fig.savefig(mdt.outp("figure_state_log.png", OUT), dpi=130); plt.show()
print(f"health: PE {pe.mean():.0f} kJ/mol (±{pe.std()/abs(pe.mean())*100:.1f}%, drift {_drift:+.2f}% over run) | "
      f"T {temp.mean():.1f}±{temp.std():.1f} K (canonical σ ≈ T√(2/Ndof)) | V spread {vol.std():.4f} nm³ (NVT: flat)")

### 2.2  Watch it happen — molecule and observables, synchronized
One play button, one timeline; the grey vertical line marks the current moment on every plot. Backbone coloured N→C; the two dashed lines are backbone α-helix hydrogen bonds, **GLN5 N–H ··· ASN1 O=C** (orange, matching its trace) and **ASP9 N–H ··· GLN5 O=C** (blue): donor N–H of residue *i* to the carbonyl O of residue *i*−4. Each line is **bold-dashed when the bond is satisfied and faint-dotted when it breaks**, where "satisfied" is the **H···O distance ≤ 2.5 Å**, a distance-only proxy (a strict H-bond criterion would add a donor–H···acceptor angle term, but distance alone is plenty to *watch* the bond come and go). Watch GLN5's orange line go dotted exactly as its orange curve spikes past the cutoff.

*Code: `player` (in `mdtviz.py`) builds the synchronized figure with **Matplotlib**'s `FuncAnimation` and embeds it as an HTML5 video; long runs are subsampled to ≤ 200 video frames while the traces stay full-resolution.*

In [ ]:
#@title §2.2 — synchronized molecule/observable player (code)
mdtviz.player(cvs[0])

### 2.3  Ray-traced filmstrip (print output, repeat 1)
Four aligned snapshots along the run, ILE4 highlighted in magenta (skips gracefully if PyMOL isn't available).

*Code: `filmstrip` (in `mdtviz.py`) ray-traces cartoon snapshots with headless open-source **PyMOL**, all superposed to frame 0; skips cleanly if PyMOL isn't installed.*

In [ ]:
#@title §2.3 — ray-traced filmstrip (code)
import matplotlib.image as mpimg
pngs, frames = mdtviz.filmstrip(cvs[0]["t"], os.path.join(OUT, "figures"))
if pngs:
    figF, axF = plt.subplots(1, len(pngs), figsize=(4 * len(pngs), 4.3), facecolor="white")
    for axf, f, fr in zip(np.atleast_1d(axF), pngs, frames):
        axf.imshow(mdtviz.sqcrop(mpimg.imread(f))); axf.axis("off"); axf.set_title(f"t = {cvs[0]['ps'][fr]} ps", fontsize=12)
    figF.suptitle("Repeat 1 filmstrip (ILE4 in magenta)", fontsize=13); figF.tight_layout()
    figF.savefig(mdt.outp("figure2_filmstrip.png", OUT), dpi=150, bbox_inches="tight"); plt.show()

### 2.4  Now the remaining independent repeats
Same start, different initial velocities: the repeats the journal asks for (live tier simulates them; reference tier just loads the rest of the shipped seeds).

In [ ]:
for k, s in enumerate(RUN_SEEDS[1:], start=1):
    if LOAD_REFERENCE:
        cvs.append(cvs_all[k])
    else:
        dcd = mdt.run_repeat(prep, s, n_prod_ps=N_PROD_PS, run_mode=RUN_MODE, out_root=OUT)
        cvs.append(mdt.compute_cvs(dcd, top=mdt.outp("stage4_minimized.pdb", PREP)))
    print(f"seed {s}: RMSD end {cvs[-1]['rmsd'][-1]:.1f} A, helix {cvs[-1]['helix'].mean():.2f}")

### 2.5  All repeats together: the spread you should expect
Six observables: two that track the whole fold, four that are local.

**The two global ones** say the fold is stable in every run. **Cα RMSD** (deviation of the backbone from the starting structure) drifts to a similar plateau in each seed, and the **radius of gyration** (overall compactness) holds steady.

**The four local ones** are more individual, and they were chosen to span contrasting timescales:

- The **indole→poly-Pro distance**, the coordinate we will pry open in 03, sits tight in its folded well, with brief excursions of a couple of ångströms in some seeds and none in others.
- The **ILE4 χ1** side-chain dihedral mostly stays in one rotamer well (g⁻, χ1 ≈ −60°) and only occasionally makes a clean, discrete hop to another (t, near 180°). This is strongly run-dependent: in three of the six reference seeds it hops to t and sits there for 8 to 20 percent of the run, and in the other three it never leaves g⁻, so in any given run you may catch a hop or nothing.
- **SER13 ψ** jitters constantly but with small amplitude.
- The **ASP9–ARG16 salt bridge** makes big, discrete excursions. The ARG16 side chain swings in and out on its long, solvent-exposed arm, so the distance jumps from about 3 Å (formed) to nearly 1 nm (fully broken). The jumps are large but their *rate* varies widely from seed to seed (across our reference runs the bridge is broken anywhere from never to roughly 20 percent of the time, and one seed never breaks it at all). That makes it the clearest reminder in the set to **never trust a single trajectory**.

The shared story: the *average* behaviour is consistent across seeds, but the exact timing of every excursion differs, and each observable lives on its own timescale. §2.6 makes that timescale statement precise. (Dihedrals are re-centered off the ±180° seam so thermal wiggles across it don't look like huge flips.)

*Code: every observable comes from `compute_cvs` (in `mdtutorial.py`): Cα RMSD, H-bond and salt-bridge distances, and the dihedrals (ψ, χ1) via **MDTraj**, helix fraction via simplified **DSSP**; Rg is MDTraj's `compute_rg`, and both dihedrals (backbone ψ, side-chain χ1) are unwrapped off the ±180° seam with `circmean_deg` / `recenter_deg`.*

In [ ]:
#@title §2.5 — all-repeats overlay (figure code)
rc = lambda a: mdt.recenter_deg(a, mdt.circmean_deg(a))
_PAL = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf"]   # 6 distinct colors, no muddy brown
fig, ax = plt.subplots(2, 3, figsize=(14, 6.6), sharex=True)
for k, c in enumerate(cvs):
    if RUN_SEEDS[k] not in PLOT_SEEDS:                     # PLOT_SEEDS declutters the overlay; every seed is still analyzed
        continue
    st = max(1, len(c["ps"]) // 2000)                     # thin dense traces so the overlay stays legible
    p = c["ps"][::st]; kw = dict(lw=0.6, alpha=0.75, color=_PAL[k % len(_PAL)], label=f"seed {RUN_SEEDS[k]}")
    ax[0, 0].plot(p, c["rmsd"][::st], **kw)
    ax[0, 1].plot(p, (md.compute_rg(c["t"]) * 10)[::st], **kw)
    ax[0, 2].plot(p, c["indolelid"][::st], **kw)
    ax[1, 0].plot(p, rc(c["ile4"])[::st], **kw)
    ax[1, 1].plot(p, rc(c["ser13"])[::st], **kw)
    ax[1, 2].plot(p, c["d9r16"][::st], **kw)
ax[0, 0].set(title="Cα RMSD", ylabel="Å"); ax[0, 1].set(title="radius of gyration", ylabel="Å")
ax[0, 2].set(title="indole→poly-Pro distance (03 cage CV)", ylabel="Å")
ax[1, 0].set(title="ILE4 χ1 — occasional rotamer hop", xlabel="time (ps)", ylabel="deg")
ax[1, 1].set(title="SER13 ψ — fast jitter", xlabel="time (ps)", ylabel="deg")
ax[1, 2].set(title="ASP9–ARG16 salt bridge (slow)", xlabel="time (ps)", ylabel="Å")
_h, _l = ax[0, 0].get_legend_handles_labels()                     # one shared legend ABOVE the grid, off the data
fig.legend(_h, _l, loc="upper center", ncol=len(cvs), fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(mdt.outp("figure2_dynamics.png", OUT), dpi=130, bbox_inches="tight"); plt.show()

### 2.6  Can we trust these averages? Convergence and uncertainty
The overlaid traces *look* consistent, but "looks consistent" is not a measurement. The number that makes it one is the **standard error of the mean (SEM)**: how far the average you just computed is likely to sit from the true average. The textbook SEM is σ/√N. That formula assumes your N frames are N *independent* draws, and they are not: consecutive frames are highly correlated. Two questions fix it.

**How many independent samples do you actually have?** Define the **integrated autocorrelation time** τ as the number of frames over which an observable remembers its earlier value. Concretely: the autocorrelation ρ(k) measures how similar the observable is to itself k frames later. It starts at 1 for k = 0 and decays toward 0 as the memory fades. Summing that decay gives **τ = ½ + Σₖ ρ(k)** (the ½ is the zero-lag self-term). Frames closer together than about 2τ are not independent, so the effective sample count is **N_eff = N ⁄ 2τ**. White noise has τ = ½ and N_eff = N; a slowly wandering observable has a large τ and a small N_eff. One practical detail: in a finite run ρ(k) turns to noise at large lag and wanders below zero, and summing that noise corrupts τ, so we stop the sum at the first crossing (the *initial-positive-sequence* cutoff; Geyer 1992, Sokal 1997).

**What is the correct error bar?** **SEM = σ ⁄ √N_eff = σ·√(2τ ⁄ N)**. The naive σ/√N is too small by the factor √(2τ).

**Estimate τ on a run long enough to see the slow motions.** τ and N are both counts of frames: τ is how long the memory lasts, N is how long you ran. A clean answer needs the run to be many correlation times long (N ⁄ τ large). The trap is that Rg's autocorrelation is a *sum* of relaxations: side chains (picoseconds), loops (nanoseconds), substate hops (nanoseconds to microseconds). A short window sees only the fast ones and τ comes out far too small. The cell below prints it for this run: on a 200 ps slice τ is tiny, on the full trajectory it is far larger. A short run *hides its own correlation time*. That is why we quote convergence from the multi-ns **reference** (`REF_ROOT`), not from the live run above.

**Why τ, and so the SEM, is a lower bound.** Two effects push the same way. First, a finite run cannot see relaxations slower than itself, so slow modes beyond your run length are simply missing from τ; run longer and τ grows until the slowest motion is captured. Second, the estimator stops at the first noisy zero-crossing, which can cut a real slow tail short before it has fully decayed. Both bias the measured τ low, so the SEM you get is best read as a floor on the true uncertainty, not a ceiling.

**The block-averaging picture, and why it cannot plateau on a run this short.** Both panels below plot a **standard error against block size**: the *error bar on the average*, not the observable levelling off. Block averaging works like this: chop the run into blocks of size *b*, average each block, and take SEM = std(block means)/√(number of blocks). Once each block is longer than the memory (*b* > τ), non-overlapping blocks are effectively independent, so the SEM climbs to a **plateau**, and that plateau is the error bar to quote (for the motions the run is long enough to see). The left panel is a **well-sampled reference case**: a toy series with a short, known τ and plenty of samples, so it does plateau. Our real run (right, Rg) is only a few correlation times long, so the curve never cleanly plateaus, and the **Flyvbjerg–Petersen error bars** on its tail balloon as the blocks run out. The error cannot be pinned down that way. With no plateau, the τ-based SEM (the red line) is the number to quote; it needs only the one trajectory. The lower grey line is the naive σ/√N, the SEM you would get if the frames were independent, as if you had **shuffled** the trajectory and destroyed its time order. The τ-based line sits above that floor by exactly √(2τ), and the block curve climbs toward it. That climb measures how correlated your run is. It does not say whether you covered the right conformations; that is the seed cross-check next.

Then the check a single trajectory can never do for itself: **do independent seeds agree within these error bars?** The cross-check below spans the reference's independent seeds. If they land on the same average within their τ-based SEMs, that is evidence of convergence on the motions the runs are long enough to see. If the seed-to-seed spread exceeds the bars, the run is undersampled no matter how tight any single bar looks.

**Where the code and the math live.** τ is `integrated_autocorr_time`, the SEM is `trust_report` (both in `mdtutorial.py`); the SEM-vs-block-size curve with its Flyvbjerg–Petersen error bars ±SEM/√(2(M−1)) is `block_curve`, and the two-panel figure that calls it is `mdtviz.convergence_figure`. τ uses Sokal's τ = ½ + Σρ definition with Geyer's initial-positive-sequence truncation (sum consecutive pairs, stop at the first non-positive pair); that truncation is what makes τ stable. Conventions and best practice: **Sokal 1997**, **Geyer 1992** (τ), **Flyvbjerg & Petersen 1989** (block averaging), **Grossfield *et al.* 2018** (uncertainty), **Chodera 2016** (equilibration detection).

In [ ]:
#@title §2.6 — convergence / block-averaging figure (code)
# Convergence is quoted from the LONG reference, never the short live run: a 200 ps window sees
# only fast motions, so its Rg autocorrelation time comes out ~50x too short (see prose). On the reference
# tier cvs IS the reference; on the live tier we load it anyway (the live tau would be untrustworthy).
if LOAD_REFERENCE:
    ref, ref_seeds = cvs, RUN_SEEDS                        # already the reference (DT set in "Get the trajectories")
    src = f"{len(ref)}-seed reference ({DT:g} ps/frame)"
else:
    try:
        mdt.ensure_reference(REF_ROOT, _BASE)              # Colab: fetch the bundle if this VM doesn't have it yet
        ref, ref_seeds, DT = mdt.load_reference(REF_ROOT)
        src = f"{len(ref)}-seed {DT * ref[0]['t'].n_frames / 1000:g} ns reference ({DT:g} ps/frame)"
    except OSError as e:                                    # FileNotFoundError + network errors (URLError) are all OSError
        ref, ref_seeds, DT = cvs, RUN_SEEDS, 1.0
        for _c, _s in zip(ref, ref_seeds): _c.setdefault("seed", _s)   # live cvs dicts lack 'seed' (only load_reference stamps it); stamp so the seed cross-check degrades instead of KeyError-ing
        src = f"{N_PROD_PS} ps LIVE run -- NO reference bundle found, so τ is biased short!"
        print("!!", e)
print("convergence source:", src)

# Why the reference, not the 200 ps live run: τ balloons as you look at more of the trajectory (see prose).
_canon = next((c for c in ref if c["seed"] == CANON_SEED), ref[0])       # the canonical example seed (config)
_rg = md.compute_rg(_canon["t"]) * 10
_ts = mdt.integrated_autocorr_time(_rg[:max(8, int(200 / DT))]) * DT     # τ from a 200 ps slice
_tf = mdt.integrated_autocorr_time(_rg) * DT                             # τ from the full run
print(f"τ(Rg) = {_ts:.0f} ps on a 200 ps slice  vs  {_tf:.0f} ps on the full {DT * len(_rg) / 1000:g} ns "
      f"({_tf / _ts:.0f}× — a short run hides its own correlation time)")

# The two-panel block-averaging figure — idealized plateau (left) vs. the real block curve with
# Flyvbjerg–Petersen error bars (right), plus the τ-based SEM floor — lives in mdtviz.convergence_figure.
# The estimator math (τ, N_eff, block SEM, the FP error bars) is in mdtutorial; see the §2.6 prose.
mdtviz.convergence_figure(md.compute_rg(_canon["t"]) * 10, dt_ps=DT, label="Reference Rg",
                          out_png=mdt.outp("figure_convergence.png", OUT)); plt.show()

# The one check a single run can't do for itself: do independent seeds agree within their τ-based bars?
print(f"\nRg per seed (mean ± τ-based SEM) — do independent seeds agree within their error bars?")
_stats, _labels = [], []
for c in ref:
    tag = f"seed {c['seed']}" + (f" / {c['solvation']}" if "solvation" in c else "")
    _labels.append(tag)
    _stats.append(mdt.trust_report(md.compute_rg(c["t"]) * 10, dt_ps=DT, label="  " + tag))
_spread = np.ptp([s["mean"] for s in _stats]); _sem = np.mean([s["sem"] for s in _stats])
print(f"\nverdict: seed-to-seed spread {_spread:.3f} Å vs a typical error bar {_sem:.3f} Å  ->  " + (
      "CONVERGED (the spread fits inside the bars)." if _spread < 2 * _sem else
      "UNDERSAMPLED (the spread is bigger than the bars, so even this run isn't long enough for Rg to this precision)."))

# the same cross-check as a glance: per-seed mean ± τ-SEM, the grand-mean agreement band, seeds clearing it flagged red
mdtviz.seed_forest(_stats, _labels, unit="Å", label="Rg", out_png=mdt.outp("figure_seed_forest.png", OUT)); plt.show()

### 2.6b  Where do those timescales live? One observable, many scales  *(an optional aside)*
**Up front: this steps off the standard path.** Everything through §2.6 is the conventional convergence toolkit: RMSD, integrated autocorrelation time, block averaging. What follows is a niche use of time-resolved wavelet analysis, itself an old standard in signal processing (Torrence & Compo 1998; Percival & Walden 2000). Instead of collapsing an observable to a single number, we read it through a **scalogram**, which shows *where in time and at which timescale* its fluctuations live. Treat this as a **coda, not a checklist item**. If you stopped at §2.6 you are on solid, conventional ground. This section is for readers who want to see *why* a single τ can mislead, resolved motion by motion.

**χ1 dihedrals.** We switch the observable to the side-chain **χ1 dihedrals** (rotation about the Cα–Cβ bond). They are small, local clocks: a single χ1 jitters within its rotamer well on picosecond timescales, hops between wells far more rarely, and inherits slower timescales still from the collective motions it couples to. A hierarchy of motions, not one clock.

**The scalogram.** A wavelet transform partitions a series' fluctuation across a (timescale × time) grid. Steady fast jitter puts power at short timescales across the whole width. A rare rotamer hop instead shows up **localized at the instant it happens**, its power spread across scales but weighted toward the slow end. Two flavors (both in `md_scalogram.py`): the **blocking / Haar DWT** (orthonormal, so an *exact* variance partition, but dyadic and blocky) and the **Morlet CWT** (continuous and smooth, at the cost of being only *near*-variance). Both answer the same *variance-partition* question. It is the same Haar machinery as §2.6, just the other half: block averaging up there used the **averaging** branch, this uses the **differencing** branch.

**The two rows, and the marginal.** Each residue's column stacks a **trace** on top and, below it, *two* scalograms of the same signal: the exact dyadic **Haar-DWT** (top) and the smooth **Morlet CWT** (bottom). Project either onto the timescale axis (right of each plot) to get the power at each scale, its **global wavelet spectrum**. The DWT row's grey bars are the exact variance per octave (Parseval); the CWT row's red curve is the smooth version. Both answer the question a single τ provokes: *does this observable's variance sit at one timescale, or spread across many?*

**Reading the strip (left → right = fast → slow).** Residues are ranked by their χ1 τ and sampled at the five **quartile boundaries**: fastest, Q1, median, Q3, slowest (five points bracketing four quartiles, not quintiles). Watch the marginal's mass shift from shorter to longer timescales as τ grows, with the dashed **2τ** line (the §2.6 number) riding up alongside it. On the left, a fast residue keeps its power low and broad with a large **N_eff**; trust that τ for the fast jitter it describes (the next cell says what it cannot see). On the right, the mass migrates into a narrow **rare-event lobe** carried by a handful of hops and N_eff collapses to a few. That single τ *could* be flagging a rare event, not a persistent slow mode: the per-residue version of §2.6's UNDERSAMPLED verdict.

In [ ]:
#@title §2.6b — scalogram figure (code)
# 2.6b needs a genuinely long trajectory to resolve slow χ1 motions; reuse the reference
# loaded in §2.6 (ref/DT). If only a short live run is present, SKIP with a note (like the PyMOL-absent
# case) rather than draw an uninterpretable 200 ps strip that a reader could mistake for signal.
import md_scalogram as msc
_ex = next((c for c in ref if c["seed"] == CANON_SEED), ref[0])         # CANON_SEED = the chosen example (config)
if _ex["t"].n_frames * DT < 2000:                                       # need >= ~2 ns to see the slow lobe
    print("§2.6b skipped: it needs the long (multi-ns) reference to resolve multi-scale χ1 dynamics.")
    print(f"  The loaded trajectory is only {_ex['t'].n_frames * DT:g} ps. Set LOAD_REFERENCE=True, or make the")
    print(f"  shipped {REF_ROOT}/ reference bundle available (see README), to render this figure.")
else:
    # Rank χ1 by circular autocorrelation time, take 5 evenly spaced fast->slow, strip their scalograms +
    # marginals. All the transform / COI / marginal math lives in md_scalogram (see prose).
    _series, _labels, _taus = msc.chi1_quartiles(_ex["t"], nq=5)
    VIEW = "both"         # which decomposition(s) to show per residue: "both" = trace + exact DWT + smooth CWT
                          #   | "dwt" = trace + exact Haar-DWT only | "cwt" = trace + smooth Morlet CWT only
    _fig = msc.marginal_strip_figure(_series, _labels, dt=DT, circular=True, obs_label="χ1 (deg)", view=VIEW,
        title=f"seed {_ex['seed']}: χ1 scalograms + marginals across τ quartile boundaries (fast → slow)")
    _fig.savefig(mdt.outp("figure_timescales.png", OUT), dpi=110, bbox_inches="tight"); plt.show()
    print("quartile boundaries (fast→slow): " + "  ".join(f"{l} τ≈{tau*DT:.0f} ps" for l, tau in zip(_labels, _taus)))

**The convergence reading.** For a well-sampled observable you would expect the power to sit at timescales much shorter than the run itself. A **slow-timescale-heavy marginal is a red flag** that the run is dominated by events it has not sampled enough of. The marginal alone cannot say which *kind*, because it is the row's summary with the time axis collapsed out: a localized hop and a persistent slow mode leave the same slow-heavy trace. The strip makes a subtler trap visible. A fast residue on the left can look cleanly converged (power low, broad, high **N_eff**) while that tidy band is really sitting **under a broad slow arch**, like the ones the slower residues show on the right, that the run was too short to raise into view. The τ is real; it describes the fast structure you *can* see, not the slow mode it may sit under. This is §2.6's "measured a τ, false impression of convergence," now residue by residue.

**Fine print.** (1) The faded **cone of influence** on the **CWT** rows is where the wavelet ran off the ends of the record into padding, not signal; do not trust power there (Torrence & Compo 1998). The exact **Haar/DWT** has the same edge effect, but there we drop any block we would have to pad to complete, so there is no cone to draw. (2) The CWT marginal is a **near-variance, qualitative** spectrum, *not* a normalized partition; for exact variance-by-scale use the blocking/Haar marginal (Parseval). (3) χ1 is an **angle**: its power is built from cos/sin, never `np.unwrap` (which would manufacture slow power out of the ±180° seam). (4) None of this resolves on a 200 ps run. The strip needs the multi-ns **reference**, and is skipped with a note if it is not present.

**Where the code and the math live.** The fast→slow residue selection is `md_scalogram.chi1_quartiles`; the strip is `md_scalogram.marginal_strip_figure`; underneath, `cwt_scalogram` / `blocking_scalogram` build the two transforms, `morlet_coi` draws the cone, and `cwt_marginal` computes the cone-respecting spectrum. Wavelet power spectrum, cone of influence, C_δ: **Torrence & Compo 1998**; the 1/s scale-bias correction: **Liu *et al.* 2007**; blocking = Haar wavelet variance: **Flyvbjerg & Petersen 1989** with **Percival & Walden 2000**.